In [1]:
%%capture
!pip install -U google-genai python-dotenv scikit-learn

In [5]:
import os
import json
import csv
import time
import numpy as np
from google import genai
from google.genai import types
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# โหลดตัวแปรจากไฟล์ .env (ระบบจะดึง GEMINI_API_KEY ไปใช้กับ genai.Client() อัตโนมัติ)
load_dotenv()

True

In [ ]:
# main class
class CoreMatchingSystem:
    def __init__(self):
        self.client = genai.Client() 
        self.model_name = "gemini-2.5-flash" 
        self.embedding_model = "gemini-embedding-001" 

    def get_vector(self, text):
        result = self.client.models.embed_content(
            model=self.embedding_model,
            contents=text,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )
        return np.array(result.embeddings[0].values).reshape(1, -1)
    
    def extract_skills_with_gemini(self, text):
        """ใช้ Gemini API สกัดทักษะสำคัญออกมาเป็น Keywords"""
        prompt = f"""
        Analyze the following Job Description and extract:
        1. Technical Skills (Programming, Tools, Frameworks)
        2. Soft Skills
        Return only a comma-separated list of keywords in English.
        
        Text: {text}
        """
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt
            )
            return response.text.strip()
        except Exception as e:
            print(f"Error extracting skills: {e}")
            return text[:500]
        
# =====================================================================
# ส่วนจำลองฐานข้อมูล: โหลดข้อมูลจากไฟล์ (รองรับทั้ง JSON และ CSV)
# =====================================================================
def load_jobs_data(file_path):
    """ฟังก์ชันสำหรับโหลดข้อมูล รองรับทั้งไฟล์ .json และ .csv"""
    ext = os.path.splitext(file_path)[-1].lower()
    
    if ext == '.json':
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
            
    elif ext == '.csv':
        jobs_data = []
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                jobs_data.append(row)
        return jobs_data
        
    else:
        raise ValueError("Unsupported file format. Please provide a .json or .csv file.")
    
def precompute_job_embeddings(input_json_file, vector_outfile, metadata_outfile, limit=900):
    matcher = CoreMatchingSystem()
    
    # 1. โหลดข้อมูล JSON
    with open(input_json_file, 'r', encoding='utf-8') as f:
        jobs_data = json.load(f)
        
    # จำกัดข้อมูลเพียง 1000 รายการแรกเพื่อไม่ให้เกิน Quota ของ Gemini API
    jobs_to_process = jobs_data[:limit]
    
    job_vectors = []
    metadata = []
    
    print(f"🚀 เริ่มประมวลผลงานจำนวน {len(jobs_to_process)} ตำแหน่ง...")
    
    for index, job in enumerate(jobs_to_process):
        # ดึงข้อมูลและดักจับกรณีชื่อ Column ตามรูปแบบใน main.ipynb
        title = job.get('Job title', job.get('Job_Title', 'Unknown Title'))
        company = job.get('Company', 'Unknown Company')
        desc = job.get('Job Description', job.get('Job_Description', ''))
        link = job.get('Job link', job.get('Job_Link', 'No Link'))
        location = job.get('Job location', job.get('Job_Location', 'Unknown Location'))
        work_arr = job.get('Work Arrangement', job.get('Work_Arrangement', 'Not Specified'))
        
        if not desc:
            continue
            
        try:
            print(f"[{index + 1}/{limit}] Embedding: {title} @ {company}")
            
            # การทำ Embedding (สามารถเลือก embbed จาก text ตรงๆ หรือใช้ extract_skills_with_gemini ก่อนก็ได้)
            # ในที่นี้ขอยกตัวอย่างเป็นการ embbed ข้อมูล text รวมเพื่อประหยัด Quota (1 งาน = 1 API Call)
            content_to_embed = f"{title}\n{desc}"
            jd_vec = matcher.get_vector(content_to_embed)
            
            # เก็บ Vector (ทำให้เป็น 1D array ก่อนเก็บ)
            job_vectors.append(jd_vec.flatten())
            
            # เก็บ Metadata สำหรับนำไปแสดงผลตอน Matching
            metadata.append({
                "Job_Title": title,
                "Company": company,
                "Job_Description": desc,
                "Job_Link": link,
                "Job_Location": location,
                "Work_Arrangement": work_arr
            })
            
            # พักระบบ 4 วินาที ป้องกัน Error 429 
            time.sleep(4)
            
        except Exception as e:
            print(f"⚠️ Error processing index {index}: {e}")
            time.sleep(10) # พักนานขึ้นถ้าเจอ Error
            continue

    # 2. บันทึกข้อมูลลงไฟล์
    np.save(vector_outfile, np.array(job_vectors))
    with open(metadata_outfile, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
        
    print("\n✨ Precompute สำเร็จ! ไฟล์ถูกสร้างเรียบร้อยแล้ว:")
    print(f"- {vector_outfile}")
    print(f"- {metadata_outfile}")

In [ ]:
# ไม่ต้องกดรันแล้ว ได้ไฟล์แล้ว
precompute_job_embeddings('enriched_jobs_with_categorization.json', 'job_vectors.npy', 'job_metadata.json')

In [19]:
def find_matching_jobs(user_cv, vector_file, metadata_file, top_k=5):
    matcher = CoreMatchingSystem()
    
    # 1. โหลดข้อมูล Vector และ Metadata ที่ Precompute ไว้
    if not os.path.exists(vector_file) or not os.path.exists(metadata_file):
        print("❌ ไม่พบไฟล์ Precompute")
        return
        
    job_vectors = np.load(vector_file)
    with open(metadata_file, 'r', encoding='utf-8') as f:
        job_metadata = json.load(f)
        
    print("⏳ กำลัง Embed ข้อมูล CV ใหม่...")
    # 2. แปลง CV ให้เป็น Vector
    cv_vec = matcher.get_vector(user_cv.strip())
    
    # 3. นำ CV Vector ไปทำ Cosine Similarity กับ Job Vectors ทั้ง 1000 งานพร้อมกัน
    similarities = cosine_similarity(cv_vec, job_vectors)[0]
    
    # 4. ดึง Index ของงานที่ได้คะแนนสูงสุด
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    print("\n" + "=" * 70)
    print(f"🏆 TOP {top_k} BEST MATCHING JOBS 🏆")
    print("=" * 70)
    
    # 5. แสดงผลลัพธ์
    for rank, idx in enumerate(top_indices):
        job = job_metadata[idx]
        score = round(similarities[idx] * 100, 2)
        
        print(f"[{rank + 1}] Match Score: {score}%")
        print(f"📌 Job Title:   {job['Job_Title']}")
        print(f"🏢 Company:     {job['Company']}")
        print(f"📍 Location:    {job['Job_Location']}")
        print(f"💼 Arrangement: {job['Work_Arrangement']}")
        print(f"🔗 Link:        {job['Job_Link']}")
        
        # ตัด Description ให้แสดงตัวอย่างสั้นๆ ป้องกันหน้าจอรกลงมาเยอะเกินไป
        short_desc = job['Job_Description'][:300].replace('\n', ' ') + "..."
        print(f"📝 Description: {short_desc}")
        print("-" * 70)

In [ ]:
# ==========================================
# Main Execute
# ==========================================
if __name__ == "__main__":
    # ขั้นตอนที่ 1: รันครั้งแรกเพื่อสร้างไฟล์ .npy และ .json
    # precompute_job_embeddings('enriched_jobs_with_categorization.json', 'job_vectors.npy', 'job_metadata.json', limit=1000)
    
    # ขั้นตอนที่ 2: เมื่อมี CV ใหม่เข้ามา (ใช้ไฟล์ที่เซฟไว้ทำงานได้อย่างรวดเร็ว)
    user_cv = """
    Experienced Developer with expertise in .NET Core, React, and PostgreSQL. 
    Strong skills in Clean Architecture and Redis. Familiar with Docker and AI tools.
    """
    find_matching_jobs(user_cv, 'job_vectors.npy', 'job_metadata.json')
#รอแค่ Quota จาก Gemini API (1000 ครั้ง) เพื่อแปลง CV เท่านั้น

⏳ กำลัง Embed ข้อมูล CV ใหม่...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 35.683196308s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-e